# Style & Occasion-Based Outfit Recommender — Ready-to-Run Notebook

This notebook is organized as a **step-by-step runnable pipeline**. Run cells from top to bottom.

**What this notebook does (end-to-end):**

1. Install Python dependencies (first-run only)
2. Validate dataset presence (you must download the Kaggle dataset and put `styles.csv` and `images/` under `data/`)
3. Load and clean metadata
4. (Optional) Compute CLIP image embeddings (can run on full dataset or a smaller sample)
5. Train a classifier on the embeddings to predict `usage` (Formal/Casual/etc.)
6. Evaluate the classifier
7. Save model artifacts for backend use

⚠️ **Important:** This notebook expects you to have the Kaggle dataset downloaded locally. See the next cell for exact instructions.


## Step 0 — Download dataset (one-time)

Download the Fashion Product Images dataset from Kaggle:

1. Go to: https://www.kaggle.com/datasets/paramaggarwal/fashion-product-images-dataset
2. Download and extract.
3. Place `styles.csv` in `data/` and the `images/` folder under `data/images/` so image `id` maps to `data/images/<id>.jpg`.

Project structure (create these folders):

```
project-root/
├─ data/
│  ├─ styles.csv
│  └─ images/   (contains files like 42431.jpg)
├─ model/
├─ backend/
└─ frontend/
```

If you prefer, use the smaller dataset `fashion-product-images-small` from the same Kaggle page for faster runs.

In [ ]:
from google.colab import userdata
import os

os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')

In [ ]:
!kaggle datasets download -d paramaggarwal/fashion-product-images-small
! unzip "fashion-product-images-small.zip" -d data

Streaming output truncated to the last 5000 lines.
  inflating: data/myntradataset/images/5813.jpg  
  inflating: data/myntradataset/images/58131.jpg  
  inflating: data/myntradataset/images/58132.jpg  
  inflating: data/myntradataset/images/58133.jpg  
  inflating: data/myntradataset/images/58135.jpg  
  inflating: data/myntradataset/images/58136.jpg  
  inflating: data/myntradataset/images/58137.jpg  
  inflating: data/myntradataset/images/58138.jpg  
  inflating: data/myntradataset/images/58139.jpg  
  inflating: data/myntradataset/images/5814.jpg  
  inflating: data/myntradataset/images/58140.jpg  
  inflating: data/myntradataset/images/58141.jpg  
  inflating: data/myntradataset/images/58143.jpg  
  inflating: data/myntradataset/images/58144.jpg  
  inflating: data/myntradataset/images/58145.jpg  
  inflating: data/myntradataset/images/58146.jpg  
  inflating: data/myntradataset/images/58147.jpg  
  inflating: data/myntradataset/images/58148.jpg  
  inflating: data/myntradataset/i

## Step 1 — Install dependencies (run once)

Run the next cell to install the required packages. If you're on Colab, this cell will install into the environment. If you're on a local machine, ensure you have Python 3.8+.

In [ ]:

# Install dependencies (may take a few minutes). Run this cell once.
import sys, subprocess, pkg_resources
required = [
    "ftfy", "regex", "tqdm", "pandas", "scikit-learn", "pillow", "fastapi", "uvicorn", "python-multipart", "joblib"
]
# specify CLIP and torch; for CPU-only environments we install cpu torch wheel via pip index (works on many systems)
# If you have CUDA, install torch following instructions on pytorch.org and skip the torch install here.
required_extra = ["git+https://github.com/openai/CLIP.git"]

def install_packages(pkgs):
    for p in pkgs:
        try:
            dist = pkg_resources.get_distribution(p.split('==')[0])
            print(f"{p} already installed: {dist.version}")
        except Exception:
            print(f"Installing {p}...")
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', p])

install_packages(required)
install_packages(required_extra)
print('\nAll pip install commands finished. If you need GPU support, please install the appropriate torch package separately.')

/tmp/ipython-input-4193269234.py:2: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import sys, subprocess, pkg_resources


Installing ftfy...
regex already installed: 2025.11.3
tqdm already installed: 4.67.1
pandas already installed: 2.2.2
scikit-learn already installed: 1.6.1
pillow already installed: 11.3.0
fastapi already installed: 0.118.3
uvicorn already installed: 0.38.0
python-multipart already installed: 0.0.20
joblib already installed: 1.5.2
Installing git+https://github.com/openai/CLIP.git...

All pip install commands finished. If you need GPU support, please install the appropriate torch package separately.


## Step 2 — Imports and configuration

Run this cell to set paths and check device (CPU/GPU). Adjust `SAMPLE_SIZE` to limit runtime (e.g., 500).

In [ ]:

import os
import time
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity
import joblib

DATA_DIR = 'data'
STYLES_CSV = os.path.join(DATA_DIR, 'styles.csv')
IMAGES_DIR = os.path.join(DATA_DIR, 'images')
MODEL_DIR = 'model'
os.makedirs(MODEL_DIR, exist_ok=True)

# To shorten run time during experimentation, set SAMPLE_SIZE to an integer (e.g., 500).
# Set SAMPLE_SIZE = None to use the entire dataset (may be slow).
SAMPLE_SIZE = 1000  # <-- change this to None to run on all items

print('DATA_DIR exists?', os.path.exists(DATA_DIR))
print('STYLES_CSV exists?', os.path.exists(STYLES_CSV))
print('IMAGES_DIR exists?', os.path.exists(IMAGES_DIR))

DATA_DIR exists? True
STYLES_CSV exists? True
IMAGES_DIR exists? True


## Step 3 — Load metadata and clean

This cell loads `styles.csv`, picks useful columns, normalizes the `usage` field, and filters rows with existing images.

In [ ]:

assert os.path.exists(STYLES_CSV), f"styles.csv not found at {STYLES_CSV}. Please download dataset and place it there as described above."
styles = pd.read_csv(STYLES_CSV, on_bad_lines='skip')
print('Raw rows:', len(styles))

# Keep only needed columns (if present)
use_cols = ['id','masterCategory','subCategory','articleType','baseColor','usage','productDisplayName']
present = [c for c in use_cols if c in styles.columns]
styles = styles[present].copy()

# Ensure id is int and drop rows missing id
styles = styles.dropna(subset=['id'])
styles['id'] = styles['id'].astype(int)

# Normalize usage label
if 'usage' in styles.columns:
    styles['usage'] = styles['usage'].astype(str).str.strip().str.title()
else:
    styles['usage'] = 'Unknown'

# Filter to items with images present
def image_exists(pid):
    return os.path.exists(os.path.join(IMAGES_DIR, f"{pid}.jpg"))

styles['has_image'] = styles['id'].apply(image_exists)
styles = styles[styles['has_image']].copy()
styles.reset_index(drop=True, inplace=True)
print('Rows with images:', len(styles))

# Optionally sample for faster runs
if SAMPLE_SIZE is not None and SAMPLE_SIZE < len(styles):
    styles = styles.sample(SAMPLE_SIZE, random_state=42).reset_index(drop=True)
    print('Using SAMPLE_SIZE:', SAMPLE_SIZE)

styles.head(5)

Raw rows: 44424
Rows with images: 44419
Using SAMPLE_SIZE: 1000


,id,masterCategory,subCategory,articleType,usage,productDisplayName,has_image
0,16947,Accessories,Eyewear,Sunglasses,Casual,Image Women Classic Eyewear Brown Sunglasses,True
1,40524,Accessories,Watches,Watches,Casual,Titan Men White Chronograph Watch,True
2,36313,Apparel,Topwear,Tshirts,Casual,Mr.Men Boys Blazing Yellow T-shirt,True
3,44188,Footwear,Flip Flops,Flip Flops,Casual,iPanema Women Black Flip Flops,True
4,33859,Footwear,Flip Flops,Flip Flops,Casual,Puma Women Lucie Pink Flip Flops,True


## Step 4 — Load CLIP model and compute image embeddings

This cell loads CLIP and computes image embeddings for the filtered items. This is the heaviest step. It will save `model/image_embeddings.npy` and `model/image_ids.csv`.

In [ ]:

import torch
import clip
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)
model, preprocess = clip.load('ViT-B/32', device=device)
model.eval()

image_ids = []
embs = []

for pid in tqdm(styles['id'].tolist(), desc='Computing embeddings'):
    p = os.path.join(IMAGES_DIR, f"{pid}.jpg")
    try:
        image = Image.open(p).convert('RGB')
        img_t = preprocess(image).unsqueeze(0).to(device)
        with torch.no_grad():
            e = model.encode_image(img_t)
            e = e.cpu().numpy()[0]
            e = e / (np.linalg.norm(e) + 1e-10)  # normalize
        image_ids.append(int(pid))
        embs.append(e)
    except Exception as exc:
        print('skip', pid, exc)

image_embeddings = np.vstack(embs)
np.save(os.path.join(MODEL_DIR, 'image_embeddings.npy'), image_embeddings)
pd.DataFrame({'id': image_ids}).to_csv(os.path.join(MODEL_DIR, 'image_ids.csv'), index=False)
print('Saved embeddings:', image_embeddings.shape)

Using device: cpu


100%|████████████████████████████████████████| 338M/338M [00:03<00:00, 109MiB/s]
Computing embeddings: 100%|██████████| 1000/1000 [04:29<00:00,  3.71it/s]

Saved embeddings: (1000, 512)


## Step 5 — Train a classifier on embeddings (predict `usage`)

Trains a Logistic Regression classifier on CLIP image embeddings to predict the `usage` label. Saves the classifier and label encoder in `model/`.

In [ ]:

# Load embeddings & ids (already saved above)
emb_path = os.path.join(MODEL_DIR, 'image_embeddings.npy')
ids_path = os.path.join(MODEL_DIR, 'image_ids.csv')
assert os.path.exists(emb_path), 'Run embeddings cell first.'
image_embeddings = np.load(emb_path)
image_ids = pd.read_csv(ids_path)['id'].tolist()

# Build y labels aligned to image_ids
styles_indexed = styles.set_index('id').loc[image_ids].reset_index()
y = styles_indexed['usage'].fillna('Unknown').astype(str)

# --- Start of fix ---
# Identify classes with only one member
usage_counts = y.value_counts()
single_instance_usages = usage_counts[usage_counts < 2].index

if not single_instance_usages.empty:
    print(f"Found {len(single_instance_usages)} usage categories with only one instance. Removing them for stratification:")
    print(single_instance_usages.tolist())
    # Filter out rows where 'usage' is in the single_instance_usages list
    styles_filtered = styles_indexed[~styles_indexed['usage'].isin(single_instance_usages)].reset_index(drop=True)

    # Re-align image_embeddings and y_enc to the filtered styles
    filtered_image_ids = styles_filtered['id'].tolist()
    # Need to get the indices of these filtered_image_ids in the original image_ids list
    original_indices = [image_ids.index(pid) for pid in filtered_image_ids]

    image_embeddings_filtered = image_embeddings[original_indices]
    y_filtered = styles_filtered['usage']
else:
    image_embeddings_filtered = image_embeddings
    y_filtered = y


le = LabelEncoder()
y_enc = le.fit_transform(y_filtered)
# --- End of fix ---

# Train-test split
# Use the filtered embeddings and labels for stratification
X_train, X_test, y_train, y_test = train_test_split(image_embeddings_filtered, y_enc, test_size=0.20, random_state=42, stratify=y_enc)

clf = LogisticRegression(max_iter=2000)
print('Training classifier...')
clf.fit(X_train, y_train)
print('Done.')

# Save artifacts
joblib.dump(clf, os.path.join(MODEL_DIR, 'usage_classifier.joblib'))
joblib.dump(le, os.path.join(MODEL_DIR, 'label_encoder.joblib'))
print('Saved classifier and label encoder to', MODEL_DIR)


Found 2 usage categories with only one instance. Removing them for stratification:
['Party', 'Smart Casual']
Training classifier...
Done.
Saved classifier and label encoder to model


## Step 6 — Evaluate classifier

Shows accuracy, classification report, and confusion matrix.

In [ ]:

# Evaluate
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print('Accuracy:', acc)
print('\nClassification report:\n', classification_report(y_test, y_pred, target_names=le.classes_))

# Confusion matrix (as dataframe)
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)
cm_df.style.background_gradient(axis=None)

Accuracy: 0.845

Classification report:
               precision    recall  f1-score   support

      Casual       0.84      1.00      0.91       158
      Ethnic       1.00      0.46      0.63        13
      Formal       1.00      0.17      0.29        12
         Nan       0.00      0.00      0.00         2
      Sports       1.00      0.20      0.33        15

    accuracy                           0.84       200
   macro avg       0.77      0.37      0.43       200
weighted avg       0.86      0.84      0.80       200



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


,Casual,Ethnic,Formal,Nan,Sports
Casual,158,0,0,0,0
Ethnic,7,6,0,0,0
Formal,10,0,2,0,0
Nan,2,0,0,0,0
Sports,12,0,0,0,3


## Step 7 — Recommendation function (use text + classifier + similarity)

This cell defines a `recommend()` function that accepts a user text query and returns top-k recommended items (id, score, metadata). It uses CLIP text embedding, classifier-predicted usage, then similarity ranking.

In [ ]:

def load_artifacts():
    emb = np.load(os.path.join(MODEL_DIR, 'image_embeddings.npy'))
    ids = pd.read_csv(os.path.join(MODEL_DIR, 'image_ids.csv'))['id'].tolist()
    clf_local = joblib.load(os.path.join(MODEL_DIR, 'usage_classifier.joblib'))
    le_local = joblib.load(os.path.join(MODEL_DIR, 'label_encoder.joblib'))
    return emb, ids, clf_local, le_local

image_embeddings, image_ids, clf_local, le_local = load_artifacts()

def recommend(user_text, top_k=5):
    # Text embedding
    tokens = clip.tokenize([user_text]).to(device)
    with torch.no_grad():
        text_emb = model.encode_text(tokens).cpu().numpy()[0]
        text_emb = text_emb / (np.linalg.norm(text_emb) + 1e-10)
    # Predict usage class from text embedding
    usage_pred = clf_local.predict(text_emb.reshape(1, -1))[0]
    usage_label = le_local.inverse_transform([usage_pred])[0]
    # Filter candidates by usage_label
    styles_map = styles.set_index('id')
    candidate_indices = [i for i,pid in enumerate(image_ids) if styles_map.loc[pid]['usage'] == usage_label]
    if len(candidate_indices) == 0:
        candidate_indices = list(range(len(image_ids)))
    candidates = image_embeddings[candidate_indices]
    sims = cosine_similarity(text_emb.reshape(1, -1), candidates)[0]
    top_idx_local = np.argsort(sims)[-top_k:][::-1]
    results = []
    for li in top_idx_local:
        gi = candidate_indices[li]
        pid = image_ids[gi]
        row = styles_map.loc[pid]
        results.append({
            'id': int(pid),
            'product': str(row.get('productDisplayName','')),
            'color': str(row.get('baseColor','')),
            'usage': str(row.get('usage','')),
            'score': float(sims[li])
        })
    return results

# Example queries
print('Example recommendations:')
print(recommend('Formal outfit for business meeting', top_k=5))

Example recommendations:
[{'id': 22376, 'product': 'Mark Taylor Men Blue & Red Striped White Shirt', 'color': '', 'usage': 'Casual', 'score': 0.2510416805744171}, {'id': 15209, 'product': 'Arrow Sport Men Solid Blue Sweater', 'color': '', 'usage': 'Casual', 'score': 0.24950215220451355}, {'id': 18759, 'product': 'Arrow Woman Aiyanna Blue Shirt', 'color': '', 'usage': 'Casual', 'score': 0.2488088607788086}, {'id': 37894, 'product': 'Happy Socks Women Black & Grey Tights', 'color': '', 'usage': 'Casual', 'score': 0.24727514386177063}, {'id': 8863, 'product': 'Mark Taylor Men Red Striped Shirt', 'color': '', 'usage': 'Casual', 'score': 0.24662169814109802}]


In [ ]:
!zip -r model.zip model/
from google.colab import files
files.download("model.zip")

  adding: model/ (stored 0%)
  adding: model/usage_classifier.joblib (deflated 5%)
  adding: model/image_embeddings.npy (deflated 7%)
  adding: model/image_ids.csv (deflated 51%)
  adding: model/label_encoder.joblib (deflated 34%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 8 — Save artifacts & next steps

Artifacts saved under `model/`:

- image_embeddings.npy
- image_ids.csv
- usage_classifier.joblib
- label_encoder.joblib

Next steps:
1. Create the FastAPI backend that loads these artifacts and exposes `/recommend?q=` endpoint.
2. Create the React frontend that calls the backend endpoint and displays images (served from backend).

If you'd like, I can now generate the full React project folder (step 2).